# Neural ODEs (NODEs) for Ballistic Motion
In this demo we will use a NODE to capture *residual* dynamics in a system of ODEs. You can use NODEs directly, but there are many demonstrations of this online, and this is a better way to use NODEs since it allows you to incorporate known physics alongside the NN.

In [ ]:
import torch
import utils

import matplotlib.pyplot as plt
import numpy as np
import pytorch_lightning as pl
import torch.nn as nn
import torch.utils.data as data

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

from pytorch_lightning import seed_everything
from torchdiffeq import odeint as td_odeint

In [ ]:
%load_ext autoreload
%autoreload 2

# 1. Fit the Dynamics with a NN

## Generate Training Data

In [ ]:
# Generate a ballistic trajectory in (x, y) to use as training data.
# Here we will use 3 balls of different masses thrown from different places with different initial conditions

In [ ]:
from scipy.integrate import odeint as sp_odeint

def ballistic(state, t, m=1.0):
    # this system is actually autonomous so does not depend on t
    x, y, vx, vy = state
    dsdt = [vx, vy, 0.0, -9.8]
    return dsdt

trajectories = []
for i, (m, s0) in enumerate(
    zip(
        [1.0, 2.0, 3.0], 
        [
            [0.0, 0.0, 10.0, 10.0], # x0, y0, vx0, vy0
            [2.0, 0.0, 5.0, 10.0], 
            [4.0, 0.0, 10.0, 5.0]
        ]
    )
):
    t = np.linspace(0, 2, 101)
    traj = sp_odeint(ballistic, s0, t, args=(m,))
    trajectories.append(
        utils.Trajectory2D(t=t, x=traj[:, 0], y=traj[:, 1], vx=traj[:, 2], vy=traj[:, 3], mass=m)
    )

In [ ]:
xm = 0
xM = 25
ym = -10
yM = 8

fig = go.Figure(
    data=[
        go.Scatter(x=[p[1] for p in traj_], y=[p[2] for p in traj_],
                     mode="lines",
                     line=dict(width=2, color=color)) for color, traj_ in zip(['blue', 'green', 'orange'], trajectories)
    ] + 
    [
        go.Scatter(x=[traj_[0][1]], y=[traj_[0][2]],
                     mode="markers",
                     marker=dict(color=color, size=10)) for color, traj_ in zip(['blue', 'green', 'orange'], trajectories)
    ]
)
fig.update_layout(
    width=600, 
    height=450,
    xaxis=dict(range=[xm, xM], autorange=False, zeroline=False),
    yaxis=dict(range=[ym, yM], autorange=False, zeroline=False),
    title_text="Kinematics", 
    title_x=0.5,
    updatemenus = [
        dict(
            type = "buttons",
            buttons = [
                dict(
                    args = [None, {"frame": {"duration": 20, "redraw": False}, "fromcurrent": True, "transition": {"duration": 20}}],
                    label = "Play",
                    method = "animate",
                    )
            ]
        )
    ]
)

fig.update(frames=[
    go.Frame(
       data=[go.Scatter(x=[traj_[k][1]], y=[traj_[k][2]]) for traj_ in trajectories],
       traces=[3+i for i in range(len(trajectories))]
   ) for k in range(len(trajectories[0]))
])

fig.show()

In [ ]:
# Create sets of t_start and t_end points for various dt, for all trajectories
X_train, y_train = utils.build_time_lagged_set(trajectories, max_incrs=3)

In [ ]:
X_train[:6]

In [ ]:
y_train[:6]

In [ ]:
len(X_train), len(y_train)

## Train a NODE

In [ ]:
torch.manual_seed(0)
# torch.set_default_dtype(dtype)
# device = torch.device(device) 

In [ ]:
seed_everything(42, workers=True)

In [ ]:
def build_loaders(
    X_train,
    y_train,
    batch_size: int = 20, 
    train_frac: float = 0.8, 
    device: str = 'cpu', 
    dtype: torch.dtype = torch.float32, 
    shuffle: bool = False, 
    drop_last: bool = True,
    seed: int = 42
) -> torch.utils.data.dataloader.DataLoader:
    train = data.TensorDataset(
        torch.tensor(X_train, dtype=dtype, requires_grad=True).to(device), 
        torch.tensor(y_train, dtype=dtype, requires_grad=True).to(device)
    )
    train_set_size = int(len(train) * train_frac)
    valid_set_size = len(train) - train_set_size

    seed = torch.Generator().manual_seed(seed)
    train_set, valid_set = data.random_split(train, [train_set_size, valid_set_size], generator=seed)

    train_loader = data.DataLoader(
        train_set, 
        batch_size=batch_size, 
        shuffle=shuffle, # Also possible because X and y are matched correctly
        drop_last=drop_last, # To avoid "spikes" from uneven dataset sizes
    )

    valid_loader = data.DataLoader(
        valid_set, 
        batch_size=batch_size, 
        shuffle=False, # Unnecessary for validation set
        drop_last=False, # No need to drop from validation set
    )

    return train_loader, valid_loader

In [ ]:
train_loader, valid_loader = build_loaders(X_train, y_train, batch_size=20, train_frac=0.8, device='cuda')

In [ ]:
class BallisticDynamics(utils.Dynamics): 
    def forward(self, t: torch.Tensor, state: torch.Tensor) -> torch.Tensor:
        """
        Compute the derivative of the state with respect to time.

        This example computes certain aspects analytically with a closed form, then uses a NN to compute a residual.

        Parameters
        ----------
        t : torch.tensor(ndim=0)
            Current time of the system's state. Ignored for autonomous systems.

        state : torch.tensor(ndim=1)
            Current system state, e.g., [x, y, x', y'].

        Returns
        -------
        deriv : torch.tensor(ndim=1)
            Derivative of system state, e.g., [x', v', x'', y''].
        """
        vx = state[2:3]
        vy = state[3:4]

        ax = torch.zeros_like(vx)
        ay = torch.zeros_like(vy)

        # "Known" part of the dynamics
        ax[:] = 0.0
        ay[:] = -8.0 # y'' = -8.0, missing the -1.8 amount = -9.8

        # "Unknown" part of the dynamics, can make dependent upon select variables
        resid = self.vector_field_resid(state) 

        # Combine to make the net dynamics
        ax += resid[0:1] # Should target 0 always
        ay += resid[1:2] # Should target -1.8 always

        return torch.cat([vx, vy, ax, ay], axis=-1)

In [ ]:
vector_field_resid = nn.Sequential(
        nn.Linear(2*2, 64), # [x, y, vx, vy] - autonomous
        nn.Tanh(), 
        nn.Linear(64, 2) # [x'', y'']
)
system = BallisticDynamics(vector_field_resid=vector_field_resid)

learner = utils.Learner(ndim=2, system=system)

trainer = pl.Trainer(
    min_epochs=1, 
    max_epochs=2, 
    accelerator="gpu",
    devices="auto",
    precision=64,
    default_root_dir="checkpoints/",
    callbacks=[pl.callbacks.early_stopping.EarlyStopping(monitor="val_loss", min_delta=0.00, patience=5, verbose=False, mode="min")],
    log_every_n_steps=1,
    enable_progress_bar=True,
    enable_checkpointing=True,
    deterministic=True,
    inference_mode=True,
    profiler=None
)

trainer.fit(
    model=learner, 
    train_dataloaders=train_loader, 
    val_dataloaders=valid_loader
)

# tensorboard --logdir .

In [ ]:
# At minimum, train for a while and monitor loss and predictive performance (plot)

# Future optimizations
# Callbacks for pl
# how to save over time and report
# make device and dtype consistent
# learnin rate updates
# profiling for bottlenecks -> num_workers in DataLoader

In [ ]:
# Check that the dynamics model is outputting ~ [0, -1.8] for all inputs
m_ = learner.system.vector_field_resid
m_.eval()
with torch.no_grad():
    pred_ = m_(torch.tensor(X_train[:, 1:-1])) # [x, y, vx, vy] is input for this model

torch.mean(pred_, axis=0) # Very close to target of [0, -1.8]!

In [ ]:
plt.plot(pred_[:, 0])
plt.plot(pred_[:, 1])
plt.axvline(891/3, color='r', alpha=0.5)
plt.axvline(891/3*2, color='r', alpha=0.5)
plt.axhline(0.0, color='k', alpha=0.5)
plt.axhline(-1.8, color='k', alpha=0.5)

In [ ]:
preds = trainer.predict(dataloaders=valid_loader)

In [ ]:
traj_ = torch.vstack(preds)
plt.plot(traj_[:, 0], traj_[:, 1], 'o', label='Predictions (Validation Set)')

cutoffs = np.array([0, 1, 2, 3], dtype=int)*len(X_train)//3
for i in range(3):
    plt.plot(X_train[cutoffs[i]:cutoffs[i+1], 1], X_train[cutoffs[i]:cutoffs[i+1], 2], '-', label=f'Ball {i+1} Observed Path')
plt.legend(loc='best')
plt.xlabel('x')
plt.ylabel('y')

# 2.  Use PySR to Represent the NN in Closed Form (see `demo/symbolic_regression`)

In [ ]:
# Here, the NN in the NODE accepts the state [x, y, vx, vy] as input so use this as training input; the output is the NN output since we are trying to mimic the 
# network. PySR seems to be setup to only regress scalar outputs so we will need to train a separate model for x'' and y'', the outputs of the NN.

# You can fit PySR with noise
# https://astroautomata.com/PySR/examples/

In [ ]:
def write_data(X, y, filename):
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1, 1)
    data = np.concatenate((X, y), axis=1)
    np.savetxt(filename, data)        

In [ ]:
# # Write the x'' and y'' outputs of the trained NN for PySR
# write_data(X_train[:, 1:-1], pred_[:, 0], filename='ax.txt')
# write_data(X_train[:, 1:-1], pred_[:, 1], filename='ay.txt')

Generally best not to do this in Jupyter - run from command line like (see `demo/symbolic_regression` for details)

~~~python
$ conda activate project-env
$ python pysr_demo.py
~~~

In [ ]:
# Observe this works pretty well!

# Need to validate: can rank based on complexity or use expressions to predict a held-out validation set and use this error to select
# Should balance (1) speed to evaluate and (2) validation accuracy

In [ ]:
# Load PySR models for [x'', and y'']
# models = PySRRegressor.from_file(...)
# eqn = models.equations_[0]['sympy_format'].iloc[7] # Select an equation
# hy_eqn = hy.from_sympy(eqn) # Use in place of hy.expression() below

In [ ]:
class System:
    def __init__(self, trajectories, radii, cooldown=0.0, ignore_terminal=False):
        self.trajectories = trajectories
        self.radii = radii
        assert len(self.trajectories) == len(self.radii)

        self.ignore_terminal = ignore_terminal
        self.cooldown = cooldown
        self._setup()

    def _setup(self):
        N = len(self.trajectories)
        
        variables = {}
        self.global_state_variable_map = {}
        for i in range(N):
            var_names = (f"x_{i}", f"y_{i}", f"vx_{i}", f"vy_{i}")
            
            x, y, vx, vy = hy.make_vars(*var_names)
            variables[i] = {'x': x, 'y': y, 'vx': vx, 'vy': vy}
        
            # Indexes the name of a variable to its location in the global state
            self.global_state_variable_map.update({v:(4*i+j) for j,v in enumerate(var_names)})

        # https://bluescarni.github.io/heyoka.py/notebooks/Event%20detection.html#terminal-events
        if self.ignore_terminal:
            callback = lambda ta, d_sgn: True
        else:
            callback = lambda ta, d_sgn: False
            
        ode_sys = []
        initial_conditions = []
        terminal_events = []
        self.event_pair = {}
        for i in range(N):
            ode_sys += [
                (variables[i]['x'], variables[i]['vx']),
                (variables[i]['y'], variables[i]['vy']),
                (variables[i]['vx'], hy.expression(0.0 + 0.0)), # Insert known + learned equation from PySR 
                (variables[i]['vy'], hy.expression(-8.0 + -1.8)), # Insert known + learned equation from PySR 
            ]
            
            initial_conditions += [
                self.trajectories[i][0][1],
                self.trajectories[i][0][2],
                self.trajectories[i][0][3],
                self.trajectories[i][0][4],
            ]

            for j in range(i+1, N):
                self.event_pair[len(self.event_pair)] = (i, j)
                terminal_events += [
                    hy.t_event(
                        (variables[i]['x'] - variables[j]['x'])**2 + (variables[i]['y'] - variables[j]['y'])**2 - ((self.radii[i] + self.radii[j])/2.0)**2,
                        cooldown=self.cooldown,
                        callback=callback
                    )
                ]

        # Create variational system for error propagation later on
        vsys = hy.var_ode_sys(ode_sys, hy.var_args.vars, order=3) # can increase order for more accuracy, but increases size of system state
        self.ta = hy.taylor_adaptive(vsys, initial_conditions, t_events=terminal_events) # compact_mode=

    def _which_event(self):
        # Then need to determine which event triggered a conjunction
        f = np.zeros(len(self.ta.t_events))
        for i, event in enumerate(self.ta.t_events):
            exp = hy.to_sympy(event.expression)
            res = float(exp.evalf(subs={symbol:self.ta.state[self.global_state_variable_map[symbol.name]] for symbol in exp.free_symbols}))
            f[i] = float(res)
    
        return np.argmin(f)

    def propagate_until(self, t_final):
        res = self.ta.propagate_until(t_final)
        
        return res

    def detect_conjunctions_before(self, t_final):
        res = self.propagate_until(t_final)
        if self.ta.time < t_final:
            print(f'Conjunction detected at t = {self.ta.time} between bodies '+'{} and {}'.format(*self.event_pair[self._which_event()]))
        
        return res

In [ ]:
s = System(trajectories, radii=[0.05, 0.05, 0.05], ignore_terminal=False)

res = s.detect_conjunctions_before(3.0)

In [ ]:
# Should set to true since multiple objects could have conjunction -> could turn into "non_terminal" and go back and count these events, if they occur

# How to reset if proposed action is taken

# Need to have tools that can iterate this way that work with risk conjunction

# Could add other equations that are R_hard < r < R_critical where R_Critical is some minimum distance of interest (this could be non-terminal)

# Three body intersection numerically ~impossible but could still be a thorny issue to deal with

In [ ]:
# Heyoka has some "simulators" - maybe we could use risk model/prediction to automatically accept proposals then integrate forward in a loop to explore options

In [ ]:
# Need to know allowable activations

# https://bluescarni.github.io/heyoka.py/notebooks/ffnn.html

In [ ]:
# Other activation functions can be built like this
def approx_gelu(x):
    return 0.5*x*(1+hy.tanh((2/np.pi)**0.5*(x+0.044715*x**3)))

In [ ]:
nn.GELU() # One solution if we want to use GELU in training is to set it to approximate for speed anyway

In [ ]:
# from typing import Any
# 
# class Convert:
#     """
#     Class with methods for converting a PyTorch NN to a Heyoka expression.
#     """
#     @staticmethod
#     def weights_and_biases_heyoka(model: nn.Module) -> NDArray[np.floating]:
#         """
#         Unroll the weights and biases from a feed-forward PyTorch model to the order that Heyoka uses.

#         Parameters
#         ----------
#         model : nn.Module
#             A feed-forward PyTorch model.

#         Returns
#         -------
#         flattened_weights : ndarray(ndim=1)
#             Weights and biases from `model`.
            
#         Notes
#         -----
#         See https://bluescarni.github.io/heyoka.py/notebooks/torch_and_heyoka.html
#         """
#         weights = {}
#         biases = {}
    
#         for name, param in model.named_parameters():
#             if "weight" in name:
#                 weights[name] = param.data.clone()
#             elif "bias" in name:
#                 biases[name] = param.data.clone()
#         biases_torch = []
#         weights_torch = []
#         for idx in range(len(weights)):
#             weights_torch.append(weights[list(weights.keys())[idx]].numpy())
#             biases_torch.append(biases[list(biases.keys())[idx]].numpy())
    
#         w_flat = []
#         b_flat = []
#         for i in range(len(weights_torch)):
#             w_flat += list(weights_torch[i].flatten())
#             b_flat += list(biases_torch[i].flatten())
#         w_flat = np.array(w_flat)
#         b_flat = np.array(b_flat)

#         return np.concatenate((w_flat, b_flat))
        
#     @staticmethod
#     def infer_structure(model: nn.Module) -> tuple[int, list[int], list[Any], int]:
#         """
#         Infer the parameters of a PyTorch feed-forward neural network for an equivalene in Heyoka.
    
#         Parameters
#         ----------
#         model : torch.nn.modules.container.Sequential
#             A sequential, feed-forward PyTorch model.

#         Returns
#         -------
#         n_in : int
#             Number of input features.

#         layer_sizes : list(int)
#             Width of each hidden layer.

#         activations : list
#             List of Heyoka functions that are activations between each hidden layer. The final entry is the output.

#         n_out : int
#             Number of outputs.
    
#         Notes
#         -----
#         An Exception is thrown if an activation is in `model` that is not supported in Heyoka. At the moment, this includes {ReLU, Sigmoid, Tanh}.
#         """
        
#         # Allowable activation functions compatible with Heyoka
#         allowed_activations = {nn.ReLU: hy.relu, nn.Sigmoid: hy.sigmoid, nn.Tanh: hy.tanh}
#         activations = []
#         layer_sizes = []
#         for layer in model:
#             if isinstance(layer, nn.Linear):
#                 layer_sizes.append(layer.out_features)
#             elif isinstance(layer, tuple(allowed_activations.keys())):
#                 for k,v in allowed_activations.items():
#                     if isinstance(layer, k):
#                         activations.append(v)
#             else:
#                 raise Exception(f'Unknown or unallowed layer found in model : {layer}')
#         activations.append(lambda inp: inp)
        
#         return model[0].in_features, layer_sizes[:-1], activations, layer_sizes[-1]

#     @staticmethod
#     def build(model: nn.Module, validate=False) -> tuple[hy.core.cfunc_dbl, list[hy.core.expression]]:
#         """
#         Convert a sequential, feed-forward PyTorch model to a Heyoka one.
        
#         Inputs
#         ------
#         model : torch.nn.modules.container.Sequential
#             A sequential, feed-forward PyTorch model.
            
#         Returns
#         -------
#         compiled_model : heyoka.core.cfunc_dbl
#             Compiled Heyoka model you can use with, e.g., numpy inputs.

#         symbolic_model : list(heyoka.core.expression)
#             An expression for each output of the neural network.
#         """
#         # Extract weights
#         flattened_weights = weights_and_biases_heyoka(model)

#         # Extract structure
#         try:
#             n_in, nn_hidden, activations, n_out = Convert.infer_structure(model)
#         except Exception as e:
#             raise Exception(f'Unable to convert model : {e}')

#         # Build Heyoka FFNN
#         inputs = hy.make_vars(*['x_'+str(i) for i in range(n_in)])
#         model_heyoka = hy.model.ffnn(
#             inputs=inputs,
#             nn_hidden=nn_hidden,
#             n_out=n_out,
#             activations=activations,
#             nn_wb=flattened_weights,
#         )
#         model_heyoka_compiled = hy.cfunc(model_heyoka, inputs)

#         if validate:
#             N = 1000
#             random_input = torch.rand((n_in, N), dtype=next(model.parameters()).dtype)
#             random_input_torch = random_input.t()
#             random_input_numpy = random_input.numpy()
#             out_array = np.zeros((n_out, N))

#             t_hey = model_heyoka_compiled(random_input_numpy, outputs=out_array)
#             t_torch = model(random_input_torch).detach().numpy().T

#             assert(np.allclose(t_hey, t_torch))

#         return model_heyoka_compiled, model_heyoka

In [ ]:
net = learner.system.vector_field_resid
model_heyoka_compiled, model_heyoka = Convert.build(net, validate=True)

In [ ]:
# now redefine system above with hy expressions replaced

In [ ]:
class SystemNN(System):
    def __init__(self, trajectories, radii, heyoka_nn, cooldown=0.0):
        self.heyoka_nn = heyoka_nn
        super().__init__(trajectories=trajectories, radii=radii, cooldown=cooldown)
        self._setup()

    def _setup(self):
        N = len(self.trajectories)
        
        variables = {}
        self.global_state_variable_map = {}
        for i in range(N):
            var_names = (f"x_{i}", f"y_{i}", f"vx_{i}", f"vy_{i}")
            
            x, y, vx, vy = hy.make_vars(*var_names)
            variables[i] = {'x': x, 'y': y, 'vx': vx, 'vy': vy}
        
            # Indexes the name of a variable to its location in the global state
            self.global_state_variable_map.update({v:(4*i+j) for j,v in enumerate(var_names)})

        """
        The main difference here is that we need to create a SEPARATE expression for each ODE because it needs to be in terms of those variables.
        Thus, a map must be defined from the variables used in heyoka_nn (see Convert.build) to the ones we are defining here in _setup().
        Otherwise everything else is the same.
        """
        ode_sys = []
        initial_conditions = []
        terminal_events = []
        self.event_pair = {}
        for i in range(N):
            ode_sys += [
                (variables[i]['x'], variables[i]['vx']),
                (variables[i]['y'], variables[i]['vy']),
                (variables[i]['vx'], hy.expression(0.0) + 
                     hy.subs(self.heyoka_nn[0], {"x_1": variables[i]['x'], "x_2": variables[i]['y'], "x_3": variables[i]['vx'], "x_4": variables[i]['vy']}) # Maps from Convert.build() names to `variables`
                ), # Insert known + learned NN
                (variables[i]['vy'], hy.expression(-8.0) + 
                     hy.subs(self.heyoka_nn[1], {"x_1": variables[i]['x'], "x_2": variables[i]['y'], "x_3": variables[i]['vx'], "x_4": variables[i]['vy']}) # Maps from Convert.build() names to `variables`
                ), # Insert known + learned NN 
            ]
            
            initial_conditions += [
                self.trajectories[i][0][1],
                self.trajectories[i][0][2],
                self.trajectories[i][0][3],
                self.trajectories[i][0][4],
            ]

            for j in range(i+1, N):
                self.event_pair[len(self.event_pair)] = (i, j)
                terminal_events += [
                    hy.t_event(
                        (variables[i]['x'] - variables[j]['x'])**2 + (variables[i]['y'] - variables[j]['y'])**2 - ((self.radii[i] + self.radii[j])/2.0)**2,
                        cooldown=self.cooldown,
                        # callback=
                    )
                ]

        self.ta = hy.taylor_adaptive(ode_sys, initial_conditions, t_events=terminal_events)

In [ ]:
s = SystemNN(trajectories, heyoka_nn=model_heyoka, radii=[0.05, 0.05, 0.05])

res = s.detect_conjunctions_before(3.0)

In [ ]:
# Real solution is more complicated because burn takes time and should account for this dynamic maneuver - maybe if too complicated can leave for higher TRL project